# Anarkali 150M live test (Colab)

Upload this notebook to Google Colab, select a T4 GPU, then run each cell from top to bottom. It downloads the recovered 150M checkpoint directly from your private Kaggle output.

In [ ]:
# Cell 1: Get Anarkali and its runtime.
!git clone --depth 1 --branch claude/distill-brio-data https://github.com/ToufiqQureshi/anarkali.git /content/anarkali
%cd /content/anarkali
!pip -q install -e .[encoder] kaggle


In [ ]:
# Cell 2: Confirm the Colab GPU.
import torch
assert torch.cuda.is_available(), 'In Colab select Runtime > Change runtime type > T4 GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Cell 3: Paste your Kaggle API token in the hidden box. No file upload is needed.
import os
from getpass import getpass
from pathlib import Path
os.environ['KAGGLE_API_TOKEN'] = getpass('Kaggle API token (hidden): ')
!kaggle kernels output toufiqqureshi/anarkali-distill-150m -p /content/anarkali-recovered -o -q --file-pattern '.*distill-run/distilled-150m/best\\.pt$'
matches = list(Path('/content/anarkali-recovered').rglob('best.pt'))
if len(matches) != 1:
    raise RuntimeError(f'Expected one 150M checkpoint, found: {matches}')
checkpoint = matches[0]
print(f'Checkpoint ready: {checkpoint} ({checkpoint.stat().st_size / 1e6:.1f} MB)')


In [ ]:
# Cell 4: Run Anarkali's built-in visible release-decision example.
!python scripts/demo_recovered_checkpoint.py --checkpoint "{checkpoint}" --device cuda


In [ ]:
# Cell 5: Test your own decision. Edit the state, question, and option text, then run.
import json
state = {
    'task': 'Release Anarkali safely.',
    'constraints': ['Do not expose credentials.', 'Run tests before publishing.'],
    'recent_actions': ['Recovered the 150M checkpoint from Kaggle.'],
}
question = {
    'type': 'choice',
    'instructions': 'What should happen next?',
    'criteria': {
        'A': 'Run the held-out evaluation and inspect its results.',
        'B': 'Publish without evaluating the recovered model.',
        'C': 'Delete saved checkpoints and train from scratch.',
    },
}
Path('/content/state.json').write_text(json.dumps(state), encoding='utf-8')
Path('/content/question.json').write_text(json.dumps(question), encoding='utf-8')
!python scripts/demo_recovered_checkpoint.py --checkpoint "{checkpoint}" --device cuda --state-json /content/state.json --question-json /content/question.json
